# Project 2 — Notebook 02: Cohort Month & Cohort Index

**Infotact Data Analytics Internship — Cohort Retention & CLTV Analysis**

**Week 1 (continued).** In Notebook 01 we combined the three files and cleaned them into
one table of **392,687** real purchases. Today (**Day 3**) we add the two columns that make
cohort analysis possible:

1. **`transaction_month`** — which month each purchase happened in.
2. **`cohort_month`** — the month of each customer's *first ever* purchase (their "starting group").
3. **`cohort_index`** — how many months *after* their first purchase a transaction happened
   (Month 0, Month 1, Month 2, …).

These three columns are the foundation of the retention matrix we will build in Week 2.

> **Reminder:** the data lives in a local `data/` folder that is **never** uploaded to GitHub.
> Only this notebook and the written summary get committed.


## What is a "cohort"? (plain English)

A **cohort** is a group of customers who *started at the same time*. In this project the start
is the month of a customer's first purchase.

- All customers whose first purchase was **December 2010** = the *December 2010 cohort*.
- If someone from that cohort buys again in January 2011, they are "retained" one month later.

To measure this we need, for every row, three facts:
- the month the purchase happened (`transaction_month`),
- the customer's first-purchase month (`cohort_month`),
- the gap between them in months (`cohort_index`).

## Step 1 — Load the clean dataset from Notebook 01

We reuse the helper we wrote on Day 2 (`src/data_prep.py`) so we get the exact same clean table in one line — no need to repeat the cleaning code.

In [ ]:
import pandas as pd
import numpy as np
from src.data_prep import load_clean_dataset

df = load_clean_dataset()
print('clean rows      :', f"{len(df):,}")
print('unique customers:', f"{df['customer_id'].nunique():,}")
print('date range      :', df['invoice_datetime'].min(), '->', df['invoice_datetime'].max())
df.head(3)

## Step 2 — Add `transaction_month`

A "month period" is just a label like `2011-03` (year + month, ignoring the day).
Pandas has a special type for this called a **Period**. We convert the full date/time into
its month using `.dt.to_period('M')`.

In [ ]:
df['transaction_month'] = df['invoice_datetime'].dt.to_period('M')
df[['invoice_datetime', 'transaction_month']].head(3)

## Step 3 — Add `cohort_month` (each customer's first-purchase month)

For every customer we find the **earliest** `transaction_month` and copy it onto *all* of
that customer's rows.

- `groupby('customer_id')` — handle each customer separately.
- `.transform('min')` — take the smallest (earliest) month and paste it back on every row
  belonging to that customer (so the column lines up with the original table).

In [ ]:
df['cohort_month'] = df.groupby('customer_id')['transaction_month'].transform('min')
df[['customer_id', 'transaction_month', 'cohort_month']].head(6)

## Step 4 — Add `cohort_index` (months since first purchase)

`cohort_index` = how many whole months passed between the first purchase and this purchase.

- Month **0** = the customer's first month.
- Month **1** = one month later, and so on.

We compute it with simple arithmetic on years and months:
`(year gap × 12) + (month gap)`.

In [ ]:
df['cohort_index'] = (
    (df['transaction_month'].dt.year  - df['cohort_month'].dt.year) * 12 +
    (df['transaction_month'].dt.month - df['cohort_month'].dt.month)
)
df[['customer_id', 'transaction_month', 'cohort_month', 'cohort_index']].head(6)

## Step 5 — Safety checks (make sure the logic is correct)

Good analysts *verify* their work. These three checks must all pass:
1. No transaction can happen **before** its cohort month → `cohort_index` never negative.
2. No missing values in the new columns.
3. The number of customers per cohort should add up to the total number of customers.

In [ ]:
neg = (df['cohort_index'] < 0).sum()
print('Rows with negative cohort_index (must be 0):', neg)
print('Missing transaction_month:', df['transaction_month'].isna().sum())
print('Missing cohort_month     :', df['cohort_month'].isna().sum())
print('cohort_index range       :', df['cohort_index'].min(), '->', df['cohort_index'].max())

cohort_sizes = df.groupby('cohort_month')['customer_id'].nunique()
print('\nSum of cohort sizes      :', cohort_sizes.sum())
print('Unique customers overall :', df['customer_id'].nunique())
print('Match?                   :', cohort_sizes.sum() == df['customer_id'].nunique())

## Step 6 — Manual spot-check on one repeat customer

Automated checks are good, but let's read one real customer's history with our own eyes to
confirm the numbers make sense. We pick the customer with the most active months.

In [ ]:
busy = df.groupby('customer_id')['transaction_month'].nunique().sort_values(ascending=False)
example_id = busy.index[0]

view = (df[df['customer_id'] == example_id]
        [['invoice_datetime', 'transaction_month', 'cohort_month', 'cohort_index']]
        .drop_duplicates('transaction_month')
        .sort_values('transaction_month')
        .head(8))
print('Customer', example_id, '— first months of activity:')
view

## Step 7 — How big is each cohort?

The size of each cohort (how many *new* customers joined that month) is the denominator we
will use in Week 2 to turn counts into retention percentages. Let's look at them now.

In [ ]:
cohort_sizes = df.groupby('cohort_month')['customer_id'].nunique()
cohort_sizes.rename('new_customers').to_frame()

## Step 8 — Save the enriched table for Week 2 (locally only)

We save the dataset *with* the new cohort columns to the local `data/` folder so Week 2's
notebook can load it instantly. **This file is git-ignored** — it stays on your device and is
never uploaded (Infotact data rule).

In [ ]:
# Period columns must be text to save cleanly to CSV; we can rebuild them later.
to_save = df.copy()
to_save['transaction_month'] = to_save['transaction_month'].astype(str)
to_save['cohort_month'] = to_save['cohort_month'].astype(str)
to_save.to_csv('data/cleaned_with_cohorts.csv', index=False)
print('Saved local file: data/cleaned_with_cohorts.csv  (git-ignored)')
print('Rows saved:', f"{len(to_save):,}")

## Step 9 — Summary & what comes next

**Done today (Day 3):**
- Added `transaction_month`, `cohort_month`, and `cohort_index` to every row.
- Verified: no negative indices, no missing values, cohort sizes sum to all 4,338 customers.
- `cohort_index` runs 0 → 12 (our 13-month window). Largest cohort = **Dec 2010 (885 customers)**.
- Saved the enriched table locally for Week 2.

**Next (start of Week 2):** use `groupby` + `pivot_table` on `cohort_month` and
`cohort_index` to build the **retention count matrix**, then divide by each cohort's size to
get the **retention percentage matrix** (the heatmap data).

> Before committing: **Kernel → Restart & Clear Output** so Git tracks code, not data blobs.